In [1]:
import sys
sys.path.append('..')

import os
import re
import glob
import pandas as pd

from nnspike.utils import extract_video_frames
from nnspike.data import create_label_dataframe, sort_by_frames_number, label_dataset_by_opencv, label_dataset_by_model, augment_dataset, set_spike_status
from nnspike.constants import  ROI_CNN

course = "right" # "right" or "left"

## Extract Frames from Videos

In [2]:
def get_all_avi_files(directory_path="C:/Users/MSAD/github/nnspike/storage/20250820/videos/", filter_timestamp=None):
    """
    Get all AVI files from the specified directory with their timestamps.
    
    Args:
        directory_path (str): Path to the directory containing AVI files
        filter_timestamp (str, optional): If set, only return files matching this timestamp pattern (supports wildcards with *)
    
    Returns:
        list: List of tuples containing (file_path, timestamp)
    """
    import os
    import glob
    import re
    import fnmatch

    # Use glob to find all .avi files in the directory
    avi_files = glob.glob(os.path.join(directory_path, "*.avi"))
    avi_files = [path.replace("\\", "/") for path in avi_files]
    
    # Sort the files for consistent ordering
    avi_files.sort()
    
    # Extract timestamps and create tuples
    result = []
    for avi_file in avi_files:
        # Extract filename without extension
        filename = os.path.basename(avi_file)
        filename_no_ext = os.path.splitext(filename)[0]
        
        # Extract timestamp from filename (assuming format: timestamp_picamera.avi)
        # This will extract the part before '_picamera'
        timestamp_match = re.match(r'^(\d{14})_.*', filename_no_ext)
        if timestamp_match:
            timestamp = timestamp_match.group(1)
        else:
            # If timestamp pattern not found, use the full filename without extension
            timestamp = filename_no_ext

        # If filter_timestamp is set, only include matching files using pattern matching
        if filter_timestamp is None or fnmatch.fnmatch(timestamp, filter_timestamp):
            result.append((avi_file, timestamp))
    
    return result

def extract_frames_from_avi_files(avi_files_with_timestamps, base_output_dir="C:/Users/MSAD/github/nnspike/storage/20250820/frames/"):
    """
    Extract frames from all AVI files and save them to folders named by timestamp.
    
    Args:
        avi_files_with_timestamps (list): List of tuples containing (file_path, timestamp)
        base_output_dir (str): Base directory where frame folders will be created
    
    Returns:
        list: List of tuples containing (output_directory, timestamp)
    """
    output_directories_with_timestamps = []
    
    for avi_file, timestamp in avi_files_with_timestamps:
        # Create output directory path
        output_dir = os.path.join(base_output_dir, timestamp)
        
        # Create directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Add trailing slash for extract_video_frames function
        output_dir_with_slash = output_dir + "/"
        
        filename = os.path.basename(avi_file)
        print(f"Extracting frames from {filename} to {output_dir_with_slash}")
        
        try:
            # Extract frames using the nnspike utility function
            extract_video_frames(avi_file, output_dir_with_slash)
            print(f"✓ Successfully extracted frames to {timestamp}/")
            # Add successful output directory and timestamp to the list
            output_directories_with_timestamps.append((output_dir, timestamp))
        except Exception as e:
            print(f"✗ Error extracting frames from {filename}: {str(e)}")
    
    return output_directories_with_timestamps



In [3]:
# Get all AVI files with their timestamps
avi_files_with_timestamps = get_all_avi_files(directory_path="C:/Users/MSAD/github/nnspike/storage/20250820/videos/")
avi_files_with_timestamps

[('C:/Users/MSAD/github/nnspike/storage/20250820/videos/20250820155306_picamera.avi',
  '20250820155306'),
 ('C:/Users/MSAD/github/nnspike/storage/20250820/videos/20250820173842_picamera.avi',
  '20250820173842'),
 ('C:/Users/MSAD/github/nnspike/storage/20250820/videos/20250820185312_picamera.avi',
  '20250820185312'),
 ('C:/Users/MSAD/github/nnspike/storage/20250820/videos/20250820185816_picamera.avi',
  '20250820185816'),
 ('C:/Users/MSAD/github/nnspike/storage/20250820/videos/20250820190608_picamera.avi',
  '20250820190608'),
 ('C:/Users/MSAD/github/nnspike/storage/20250820/videos/20250820190702_picamera.avi',
  '20250820190702'),
 ('C:/Users/MSAD/github/nnspike/storage/20250820/videos/20250820190935_picamera.avi',
  '20250820190935')]

In [4]:
    # Extract frames from all AVI files
if avi_files_with_timestamps:
    print("\nStarting frame extraction...")
    output_dirs_with_timestamps = extract_frames_from_avi_files(avi_files_with_timestamps, base_output_dir="C:/Users/MSAD/github/nnspike/storage/20250820/frames/")
    print("\nFrame extraction completed!")
    print(f"Successfully created {len(output_dirs_with_timestamps)} output directories:")
    for output_dir, timestamp in output_dirs_with_timestamps:
        print(f"  - {output_dir} (timestamp: {timestamp})")
else:
    print("No AVI files found to process.")
    output_dirs_with_timestamps = []
    
output_dirs_with_timestamps


Starting frame extraction...
Extracting frames from 20250820155306_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820155306/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\20250820\frames\20250820155306
✓ Successfully extracted frames to 20250820155306/
Extracting frames from 20250820173842_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820173842/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\20250820\frames\20250820173842
✓ Successfully extracted frames to 20250820173842/
Extracting frames from 20250820185312_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820185312/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\20250820\frames\20250820185312
✓ Successfully extracted frames to 20250820185312/
Extracting frames from 20250820185816_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820185816/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\

[('C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820155306',
  '20250820155306'),
 ('C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820173842',
  '20250820173842'),
 ('C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820185312',
  '20250820185312'),
 ('C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820185816',
  '20250820185816'),
 ('C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820190608',
  '20250820190608'),
 ('C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820190702',
  '20250820190702'),
 ('C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820190935',
  '20250820190935')]

In [5]:
for output_dir, timestamp in output_dirs_with_timestamps:
    print(f"Output directory: {output_dir} (timestamp: {timestamp})")
    
    course = "right"
    label_df = create_label_dataframe(output_dir +"/*", course)
    label_df = sort_by_frames_number(label_df)
    label_df = label_dataset_by_opencv(label_df, ROI_CNN, 80)

    status_df = pd.read_csv(f"C:/Users/MSAD/github/nnspike/storage/20250820/sensor_data/{timestamp}_sensor_log.csv")
    df = set_spike_status(label_df, status_df)

    # Export to a csv file
    df.to_csv(f"C:/Users/MSAD/github/nnspike/storage/20250820/labels/{timestamp}_label.csv", index=False)

Output directory: C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820155306 (timestamp: 20250820155306)


Processing: 100%|██████████| 1998/1998 [00:38<00:00, 51.47it/s]


Output directory: C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820173842 (timestamp: 20250820173842)


Processing: 100%|██████████| 1711/1711 [00:33<00:00, 51.57it/s]


Output directory: C:/Users/MSAD/github/nnspike/storage/20250820/frames/20250820185312 (timestamp: 20250820185312)


Processing: 100%|██████████| 2110/2110 [00:38<00:00, 54.14it/s]


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/MSAD/github/nnspike/storage/20250820/sensor_data/20250820185312_sensor_log.csv'